# 13 — PTA analytical model matching (beyond the Horner straight line)

Code: `src/pta_model.py`. Modern well-test software fits a full reservoir model to the **whole** build-up,
not just the straight-line part. Model = radial flow to a vertical well with
- **wellbore storage C** [bbl/psi] — the early unit slope + hump,
- **skin S**,
- an **outer boundary**: infinite | constant-pressure circle (aquifer / injector support) | closed circle (no flow).

Solved exactly in Laplace space (Van Everdingen & Hurst, with Bessel functions), inverted with Stehfest,
build-up by superposition: `dP(dt) = 141.2·q·B·μ/(k·h) · [pD(dt) − pD(tp+dt) + pD(tp)]`.
Fitted by least squares on log(dP) (relative errors) → `k`, `S`, `C`, boundary radius.

**Model choice** — a boundary model is accepted only if (1) AIC improves by > 4 vs infinite
(AIC = fit quality with a penalty per parameter) AND (2) the boundary is inside the radius of investigation
`r_i = √(k·t/(948·φ·μ·ct))` (the pressure wave actually reached it during the test).

**Checks of the solver** (`pwd` vs textbook limits): infinite late time `½(ln tD + 0.809)` ✓, storage
`pD = tD/CD` ✓, skin adds `S` ✓, closed circle pseudo-steady slope `2/reD²` ✓, constant pressure → `ln reD` ✓.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import pta
import pta_model as M

# solver sanity checks
t = np.array([1e4, 1e6])
print('infinite  :', M.pwd(t, 0, 0, 0, 'infinite').round(4), ' textbook:', (0.5 * (np.log(t) + 0.80907)).round(4))
print('const-P   :', M.pwd(np.array([1e9]), 0, 0, 1000, 'constant_pressure').round(4), ' textbook ln(1000):', round(np.log(1000), 4))

OUT = load.ROOT / 'outputs'
hf = load.load_high_freq_clean()
pvt = load.load_pvt().set_index('well_name')
completion = load.load_completion().set_index('well_name')
vfm_rates = pd.read_excel(OUT / 'VFM_predictions.xlsx', sheet_name='predictions')
vfm_rates['date'] = pd.to_datetime(vfm_rates.date)

In [ ]:
rows, all_fits = [], []
for _, event in pta.find_shut_ins(hf).iterrows():
    rates = vfm_rates[vfm_rates.well_name == event.well_name]
    q = rates[rates.date == event.t_shut.normalize() - pd.Timedelta(days=1)].oil_rate_stbd.iloc[0]
    cum_oil = completion.loc[event.well_name, 'cum_oil_at_2025_01_01_stb'] + \
              rates[rates.date < event.t_shut.normalize()].oil_rate_stbd.sum()
    p = pvt.loc[event.well_name]
    horner = pta.analyze_event(hf, event, q, cum_oil, p)
    fits, best, dt, dp = M.fit_all_models(horner, p)

    fig = M.plot_model_match(horner, fits, best, dt, dp)
    fig.savefig(OUT / 'plots' / f"pta_model_{event.well_name}_{event.t_shut:%Y%m%d}.png", dpi=110)
    plt.show()

    for f in fits:
        all_fits.append({'well_name': event.well_name, 'boundary': f['boundary'], 'chosen': f is best,
                         'k_md': f['k_md'], 'skin': f['skin'], 'C_bbl_psi': f['C_bbl_psi'],
                         'boundary_radius_ft': f['boundary_radius_ft'],
                         'radius_investigation_ft': f['radius_investigation_ft'],
                         'rms_error_pct': f['rms_error_pct'], 'aic': f['aic']})
    rows.append({'well_name': event.well_name, 't_shut': event.t_shut, 'model': best['boundary'],
                 'k_model': best['k_md'], 'k_horner': horner['k_md'],
                 'skin_model': best['skin'], 'skin_horner': horner['skin'],
                 'C_bbl_psi': best['C_bbl_psi'], 'boundary_radius_ft': best['boundary_radius_ft'],
                 'radius_investigation_ft': best['radius_investigation_ft'], 'rms_error_pct': best['rms_error_pct']})

pd.DataFrame(all_fits).round(3)

## Model match vs Horner

In [ ]:
summary = pd.DataFrame(rows)
summary['k_model_vs_horner_%'] = (summary.k_model / summary.k_horner - 1) * 100
summary.to_csv(OUT / 'pta_model_matching.csv', index=False)
summary.round(2)

**Conclusions**
- The model matches the whole build-up (storage hump, radial flow, late time) within **0.02–0.6 %** RMS.
- **k and skin agree with Horner** (k within ~0–5 %, skin within ~0.5): two independent methods, same answer.
  The deliverable keeps the Horner values (what the challenge asks for); the model match is the cross-check.
- **New information Horner cannot give:**
  - wellbore storage C ≈ 0.01–0.03 bbl/psi for all wells (consistent with the tubing volume);
  - **GL-01: constant-pressure support at ~700 ft** — infinite model clearly fails at late time (derivative
    drops); fits its high, rising water cut (aquifer / water injection support);
  - GL-10: constant-pressure model also preferred, but the boundary (576 ft) is right at the edge of the radius
    of investigation → *possible* support, to be confirmed by a longer test;
  - GL-02, GL-04, GL-06, GL-08: no boundary seen during the test (infinite-acting).
- Note: GL-04's closed model has a better AIC and its radius (2,850 ft) matches the drainage area (2,736 ft),
  but it lies beyond the 48-h radius of investigation (~900 ft), so the conservative rule keeps 'infinite'.